# 44 — SHD / real-spike walkthrough (local artefacts only)

Visual tour of **local** Spiking Heidelberg Digits (SHD) and Vertex training
artefacts already checked into this repository. Fail closed when files are
missing — no network download in this notebook.

## Honesty box

| | |
|---|---|
| **Proves** | Local SHD H5 / Vertex `training_log.csv` can be listed and plotted when present; basic spike-density or loss curves from those files. |
| **Does not prove** | That every intended SHD seed is present, external leaderboard acceptance, FPGA deployment of the trained net, or polyglot neuron parity. |
| **Artefacts** | Reads under `data/masquelier_shd/` and `results/vertex/` only. |
| **Models** | Training logs may refer to standard LIF / DCLS stacks; optional HF AdEx comparison is synthetic, not SHD-trained. |


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
if not (REPO / "data").exists() and (REPO.parent / "data").exists():
    REPO = REPO.parent

SHD_H5 = [
    REPO / "data/masquelier_shd/neuromorphic_training-main/Datasets/SHD/shd_train.h5",
    REPO / "data/masquelier_shd/neuromorphic_training-main/Datasets/SHD/extract/shd_train.h5",
]
VERTEX_ROOT = REPO / "results/vertex"
print("REPO", REPO)


## 1. Discover local artefacts (fail closed)


In [ ]:
def first_existing(paths: list[Path]) -> Path | None:
    for p in paths:
        if p.is_file():
            return p
    return None

h5_path = first_existing(SHD_H5)
logs = sorted(VERTEX_ROOT.rglob("training_log.csv")) if VERTEX_ROOT.is_dir() else []
print("SHD H5:", h5_path if h5_path else "MISSING — SHD spike section will skip")
print(f"Vertex training logs found: {len(logs)}")
for p in logs[:8]:
    print(" ", p.relative_to(REPO))
if h5_path is None and not logs:
    raise SystemExit("No local SHD/Vertex artefacts — stop (fail closed). Checkout full data/results trees.")


## 2. Optional: peek SHD train H5 (if h5py + file present)


In [ ]:
if h5_path is None:
    print("Skip SHD H5 peek.")
else:
    try:
        import h5py
    except ImportError:
        print("h5py not installed — skip H5 peek (optional extra).")
    else:
        with h5py.File(h5_path, "r") as f:
            print("top keys:", list(f.keys())[:20])
            # common SHD layout: spikes/times, spikes/units, labels
            def _walk(name, obj):
                if isinstance(obj, h5py.Dataset) and len(getattr(obj, "shape", ())) <= 2:
                    print(f"  dataset {name}: shape={obj.shape} dtype={obj.dtype}")
            f.visititems(_walk)
        # lightweight sample plot if times/units exist
        with h5py.File(h5_path, "r") as f:
            times = units = None
            for cand_t, cand_u in [
                ("spikes/times", "spikes/units"),
                ("times", "units"),
            ]:
                if cand_t in f and cand_u in f:
                    times = f[cand_t]
                    units = f[cand_u]
                    break
            if times is not None and hasattr(times, "__getitem__"):
                # variable-length: try first sample
                try:
                    t0 = np.asarray(times[0]).ravel()
                    u0 = np.asarray(units[0]).ravel()
                    n = min(len(t0), 2000)
                    fig, ax = plt.subplots(figsize=(9, 3))
                    ax.scatter(t0[:n], u0[:n], s=2, alpha=0.5)
                    ax.set_title(f"SHD sample 0 raster (first {n} events) — {h5_path.name}")
                    ax.set_xlabel("time")
                    ax.set_ylabel("unit")
                    fig.tight_layout()
                    plt.show()
                except Exception as exc:
                    print("Could not plot sample 0:", type(exc).__name__, exc)
            else:
                print("No standard spikes/times layout found — metadata-only walk.")


## 3. Vertex training_log.csv (accuracy / loss curves when present)


In [ ]:
if not logs:
    print("No training_log.csv under results/vertex — skip curves.")
else:
    # pick first log with >2 rows
    chosen = None
    for p in logs:
        try:
            import csv
            with p.open() as fh:
                rows = list(csv.DictReader(fh))
            if len(rows) >= 2:
                chosen = (p, rows)
                break
        except Exception:
            continue
    if chosen is None:
        print("Logs unreadable or empty.")
    else:
        p, rows = chosen
        keys = list(rows[0].keys())
        print("using", p.relative_to(REPO), "cols", keys)
        # prefer accuracy-like columns
        ykey = next((k for k in keys if "acc" in k.lower()), None)
        if ykey is None:
            ykey = next((k for k in keys if "loss" in k.lower()), keys[-1])
        xkey = next((k for k in keys if "epoch" in k.lower() or "step" in k.lower()), keys[0])
        xs, ys = [], []
        for r in rows:
            try:
                xs.append(float(r[xkey]))
                ys.append(float(r[ykey]))
            except (TypeError, ValueError, KeyError):
                continue
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(xs, ys, "-o", ms=3)
        ax.set_xlabel(xkey)
        ax.set_ylabel(ykey)
        ax.set_title(f"Local Vertex log: {p.parent.name}")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()
print("NB-44 complete (local artefacts only).")
